# 🏋️ Exercise 4 — SQL
**Tujuan:** menjawab pertanyaan bisnis menggunakan SQL.

> Isi bagian bertanda `# TODO`, lalu jalankan sel **Cek Jawaban**.

## ⚙️ Persiapan
Pastikan kernel **.venv** sudah dipilih (pojok kanan atas) dan file `.env` sudah diisi, lalu jalankan sel-sel berikut.

In [1]:
# ==== KONFIGURASI ====
# Kredensial database dibaca dari file .env di root repo (salin .env.example → .env, isi dari kartu akses).
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

GITHUB_USER = "AisahAlfiyatusR"     # akun GitHub pemateri (repo asal yang Anda fork)
REPO        = "data-science-workshop"
RAW_BASE    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main"
API_URL     = os.getenv("API_URL") or RAW_BASE + "/api/static/sales_page_{page}.json"

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = int(os.getenv("DB_PORT", 5432))
DB_NAME     = os.getenv("DB_NAME", "workshop")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
SCHEMA      = DB_USER   # setiap peserta punya schema pribadi bernama sama dengan username

if not (DB_HOST and DB_USER and DB_PASSWORD):
    print("⚠️ File .env belum ditemukan/diisi — bagian database tidak akan berjalan.")
else:
    print(f"Konfigurasi OK — login database sebagai {DB_USER} @ {DB_HOST}")

Konfigurasi OK — login database sebagai peserta25 @ ep-wandering-math-b3bml8w1-pooler.c-4.ap-southeast-1.aws.neon.tech


In [2]:
import os, requests
import pandas as pd
from sqlalchemy import create_engine, text, Date
from sqlalchemy.engine import URL
pd.set_option("display.max_columns", 20)


def page_url(base, page, page_size=100):
    """URL satu halaman — mendukung API statis ('{page}' di URL) maupun API server (?page=)."""
    if "{page}" in base:
        return base.format(page=page)
    return f"{base}?page={page}&page_size={page_size}"

In [3]:
engine = create_engine(URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
                                  host=DB_HOST, port=DB_PORT, database=DB_NAME),
                       pool_size=1, max_overflow=1, pool_pre_ping=True)   # hemat koneksi: ±70 peserta memakai server yang sama
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_user, current_schema()")).fetchone())

('peserta25', 'peserta25')


In [4]:
def run_sql(query):
    """Menjalankan query SELECT dan mengembalikan DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

In [5]:
TABLE = f"{SCHEMA}.sales_clean"   # ganti ke "ref.sales_clean" jika Load Anda belum berhasil

Daftar soal juga tersedia di `08_sql_exercise/Latihan_SQL.sql` — bisa dikerjakan di DBeaver/pgAdmin.

### Soal 1
10 transaksi terbaru (urutkan dari tanggal paling baru).

In [6]:
q1 = f"""
    SELECT *
    FROM {TABLE}
    ORDER BY transaction_date DESC
    LIMIT 10
"""

r = run_sql(q1)
r

,transaction_id,transaction_date,customer_id,city,product_id,product_name,category,quantity,unit_price,total_amount
0,TRX00196,2026-03-31,C041,Surabaya,P01,Paket Internet 20 Mbps,Internet,1,250000,250000
1,TRX00069,2026-03-31,C002,Surabaya,P06,Biaya Instalasi,Layanan,1,150000,150000
2,TRX00254,2026-03-30,C035,Sidoarjo,P02,Paket Internet 50 Mbps,Internet,3,400000,1200000
3,TRX00159,2026-03-30,C016,Surabaya,P02,Paket Internet 50 Mbps,Internet,1,400000,400000
4,TRX00103,2026-03-30,C030,Malang,P01,Paket Internet 20 Mbps,Internet,1,250000,250000
5,TRX00116,2026-03-30,C002,Surabaya,P07,Upgrade Bandwidth,Layanan,1,200000,200000
6,TRX00190,2026-03-30,C026,Pasuruan,P02,Paket Internet 50 Mbps,Internet,2,400000,800000
7,TRX00077,2026-03-30,C035,Sidoarjo,P07,Upgrade Bandwidth,Layanan,1,200000,200000
8,TRX00212,2026-03-30,C049,Surabaya,P04,Router WiFi,Perangkat,1,450000,450000
9,TRX00287,2026-03-30,C049,Surabaya,P02,Paket Internet 50 Mbps,Internet,3,400000,1200000


In [7]:
r = run_sql(q1)
assert len(r) == 10 and str(r["transaction_date"].iloc[0]) == "2026-03-31", "Belum tepat, coba lagi"
print("✅ Soal 1 benar")

✅ Soal 1 benar


### Soal 2
Total revenue per bulan.

In [8]:
q2 = f"""
    SELECT
        DATE_TRUNC('month', transaction_date)::date AS bulan,
        SUM(total_amount) AS total_revenue
    FROM {TABLE}
    GROUP BY DATE_TRUNC('month', transaction_date)::date
    ORDER BY bulan ASC
"""

r = run_sql(q2)
r

,bulan,total_revenue
0,2026-01-01,35850000.0
1,2026-02-01,51800000.0
2,2026-03-01,58700000.0


In [9]:
r = run_sql(q2)
assert len(r) == 3 and [int(x) for x in r.iloc[:, 1]] == [35850000, 51800000, 58700000], "Belum tepat, coba lagi"
print("✅ Soal 2 benar")

✅ Soal 2 benar


### Soal 3
Ranking kota berdasarkan total revenue (tertinggi di atas).

In [10]:
q3 = f"""
    SELECT
        city,
        SUM(total_amount) AS total_revenue
    FROM {TABLE}
    GROUP BY city
    ORDER BY total_revenue DESC
"""

r = run_sql(q3)
r

,city,total_revenue
0,Surabaya,58850000.0
1,Malang,35700000.0
2,Sidoarjo,24500000.0
3,Batu,17050000.0
4,Pasuruan,10250000.0


In [11]:
r = run_sql(q3)
assert r.iloc[0, 0] == "Surabaya" and int(r.iloc[0, 1]) == 58850000 and len(r) == 5, "Belum tepat, coba lagi"
print("✅ Soal 3 benar")

✅ Soal 3 benar


### Soal 4
Produk terlaris berdasarkan total quantity.

In [12]:
q4 = f"""
    SELECT
        product_name,
        SUM(quantity) AS total_quantity
    FROM {TABLE}
    GROUP BY product_name
    ORDER BY total_quantity DESC
    LIMIT 1
"""

r = run_sql(q4)
r

,product_name,total_quantity
0,Paket Internet 50 Mbps,98.0


In [13]:
r = run_sql(q4)
assert r.iloc[0, 0] == "Paket Internet 50 Mbps" and int(r.iloc[0, 1]) == 98, "Belum tepat, coba lagi"
print("✅ Soal 4 benar")

✅ Soal 4 benar


### Soal 5
Revenue per kategori beserta persentase kontribusinya (2 desimal).

In [14]:
q5 = f"""
    SELECT
        category,
        SUM(total_amount) AS total_revenue,
        ROUND(
            100.0 * SUM(total_amount) / SUM(SUM(total_amount)) OVER (),
            2
        ) AS contribution_pct
    FROM {TABLE}
    GROUP BY category
    ORDER BY total_revenue DESC
"""

r = run_sql(q5)
r

,category,total_revenue,contribution_pct
0,Internet,88850000.0,60.71
1,Perangkat,38550000.0,26.34
2,Layanan,18950000.0,12.95


In [15]:
r = run_sql(q5)
assert r.iloc[0, 0] == "Internet" and abs(float(r.iloc[0, 2]) - 60.71) < 0.01, "Belum tepat, coba lagi"
print("✅ Soal 5 benar")

✅ Soal 5 benar


### Soal 6
Top 5 pelanggan berdasarkan total belanja.

In [16]:
q6 = f"""
    SELECT
        customer_id,
        SUM(total_amount) AS total_belanja
    FROM {TABLE}
    GROUP BY customer_id
    ORDER BY total_belanja DESC
    LIMIT 5
"""

r = run_sql(q6)
r

,customer_id,total_belanja
0,C013,6250000.0
1,C011,5100000.0
2,C008,4450000.0
3,C026,4350000.0
4,C014,4100000.0


In [17]:
r = run_sql(q6)
assert len(r) == 5 and r.iloc[0, 0] == "C013" and int(r.iloc[0, 1]) == 6250000, "Belum tepat, coba lagi"
print("✅ Soal 6 benar")

✅ Soal 6 benar


### Soal 7
Jumlah transaksi dan rata-rata nilai transaksi per kota, hanya kota dengan > 30 transaksi.

In [18]:
q7 = f"""
    SELECT
        city,
        COUNT(*) AS jumlah_transaksi,
        ROUND(AVG(total_amount), 2) AS rata_rata_transaksi
    FROM {TABLE}
    GROUP BY city
    HAVING COUNT(*) > 30
    ORDER BY jumlah_transaksi DESC
"""

r = run_sql(q7)
r

,city,jumlah_transaksi,rata_rata_transaksi
0,Surabaya,127,463385.83
1,Malang,74,482432.43
2,Sidoarjo,46,532608.70
3,Batu,34,501470.59


In [19]:
r = run_sql(q7)
assert set(r.iloc[:, 0]) == {"Surabaya", "Malang", "Sidoarjo", "Batu"}, "Belum tepat, coba lagi"
print("✅ Soal 7 benar")

✅ Soal 7 benar


### Soal 8
(Bonus) Pertumbuhan revenue month-over-month (%) menggunakan window function LAG().

In [20]:
q8 = f"""
    WITH monthly_revenue AS (
        SELECT
            DATE_TRUNC('month', transaction_date)::date AS bulan,
            SUM(total_amount) AS revenue
        FROM {TABLE}
        GROUP BY DATE_TRUNC('month', transaction_date)::date
    ),
    revenue_comparison AS (
        SELECT
            bulan,
            revenue,
            LAG(revenue) OVER (ORDER BY bulan) AS revenue_bulan_lalu
        FROM monthly_revenue
    )
    SELECT
        bulan,
        revenue,
        ROUND(
            100.0 * (revenue - revenue_bulan_lalu)
            / NULLIF(revenue_bulan_lalu, 0),
            2
        ) AS pertumbuhan_mom_pct
    FROM revenue_comparison
    ORDER BY bulan
"""

r = run_sql(q8)
r

,bulan,revenue,pertumbuhan_mom_pct
0,2026-01-01,35850000.0,NaN
1,2026-02-01,51800000.0,44.49
2,2026-03-01,58700000.0,13.32


In [21]:
r = run_sql(q8)
assert len(r) == 3 and abs(float(r.iloc[1, -1]) - 44.49) < 0.01, "Belum tepat, coba lagi"
print("✅ Soal 8 benar")

✅ Soal 8 benar
